In [56]:
import pandas as pd 
import numpy as np 
import sklearn 
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, recall_score, f1_score, confusion_matrix, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


In [9]:
df = pd.read_csv('./data/S05-hw-dataset.csv')

print("\nПервые строки датасета (head):\n", df.head())
print("\nИнформация о столбцах и типах (info):\n")
df.info()
print("\nБазовые описательные статистки для числовых признаков (describe):\n", df.describe())
print("\nРаспределение целевого признака:\n", df["default"].value_counts(normalize=True))


Первые строки датасета (head):
    client_id  age  income  years_employed  credit_score  debt_to_income  \
0          1   25   94074              22           839        0.547339   
1          2   58   51884              26           565        0.290882   
2          3   53   48656              39           561        0.522340   
3          4   42   81492              30           582        0.709123   
4          5   42   94713               8           642        0.793392   

   num_credit_cards  num_late_payments  has_mortgage  has_car_loan  \
0                 1                  7             0             0   
1                 1                  1             0             1   
2                 1                 13             0             0   
3                 2                 10             1             1   
4                 3                  3             0             0   

   savings_balance  checking_balance  region_risk_score  \
0            26057              5229

## Краткая фиксация наблюдений

- В датасете порядка 3000 обьектов и 17 признаков(из которых всего 2 вещественных, остальные - целые числа), среди обьектов пропусков не замеченно 

- Явные аномалии присутствуют, например у клиента с id = 1, его возраст равен 25 годам, а стаж - 22, это невозможно, следовательно является аномалией, или же у клиента с id = 3, его просрочка по платежу равна 13 годам, хотя клиентом банка он является только 5 лет

- Таргет распределен примерно сбалансированно, класс 0 - 59%, 1 - 41%

In [17]:
#Выделение матрицы признаков X и таргета y
y = df['default']
X = df[[x for x in df.columns if not 'id' in x and not 'default' in x]]


In [21]:
#Простая предобработка 

#проверяем что признаки числовые 
from pandas.api.types import is_numeric_dtype

for col in X: 
    if(not is_numeric_dtype(X[col])):
        print(X[col])
        
col = X["debt_to_income"]

#проверяем что значения в debt_to_income между 0 и 1 
check = col.between(0,1)
all_check = check.all()
print(all_check)

True


In [50]:
#Создаем обучающие и тестовые выборки 

from sklearn.model_selection import train_test_split, GridSearchCV

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.25, random_state=42, stratify=y_temp)

In [39]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="stratified", random_state=42)

#Обучаем на выборках 
baseline.fit(X_train, y_train)

y_val_pred_baseline = baseline.predict(X_val)
y_val_proba_baseiline = baseline.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, y_val_pred_baseline))
print("Precision:", precision_score(y_val, y_val_pred_baseline, zero_division=0))
print("Recall:", recall_score(y_val, y_val_pred_baseline, zero_division=0))
print("F1 Score:", f1_score(y_val, y_val_pred_baseline, zero_division=0))


try: 
    print("ROC-AUC:", roc_auc_score(y_val, y_val_proba_baseiline))
except ValueError as e:
    print("Ошибка подсчета ROC-AUC:",e) 
    

    

Accuracy: 0.5133333333333333
Precision: 0.4108527131782946
Recall: 0.43089430894308944
F1 Score: 0.42063492063492064
ROC-AUC: 0.5007578889348215


## Что делает Baseline 

Задает понимания минимума качества, путем показания того, какого результата можно добиться, самым простым способом и от этого уже идет нарост фич 

In [69]:


log_reg_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000))
])

params_grid = {
    "logreg__C": [0.01, 0.1, 1.0, 10.0],
}

grid = GridSearchCV(
    estimator=log_reg_pipeline,
    param_grid=params_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
)

grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best score on CV:", grid.best_score_)

best_model = grid.best_estimator_

y_val_pred = best_model.predict(X_test)
y_val_proba = best_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_val_pred, zero_division=0))
print("F1:", f1_score(y_val, y_val_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_val, y_val_proba))
print("Confussion matrix:", confusion_matrix(y_val, y_val_pred))

Best params: {'logreg__C': 0.1}
Best score on CV: 0.853594897701044
Accuracy: 0.53
Precision: 0.4142857142857143
Recall: 0.6869918699186992
F1: 0.3815789473684211
ROC-AUC: 0.5062009094667218
Confussion matrix: [[231 123]
 [159  87]]


In [58]:
import matplotlib.pyplot as plt


fpr, tpr, _ = roc_curve(y_val, y_val_proba)

plt.figure(figsize=(6, 6))

plt.plot(fpr, tpr, label="ROC curve")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve – LogisticRegression")
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig("./figures/ROC_curve.png", dpi=150, bbox_inches="tight")
plt.close()

In [66]:
results = pd.DataFrame(columns=["Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"])

results.loc[len(results)] = ["Dummy", 
                             accuracy_score(y_val, y_val_pred_baseline),  
                             precision_score(y_val, y_val_pred_baseline, zero_division=0), 
                             recall_score(y_val, y_val_pred_baseline, zero_division=0), 
                             f1_score(y_val, y_val_pred_baseline, zero_division=0),
                             roc_auc_score(y_val, y_val_proba_baseiline)
                             ]

results.loc[len(results)] = ["LogisticRegression", 
                             accuracy_score(y_val, y_val_pred), 
                             precision_score(y_val, y_val_pred, zero_division=0), 
                             recall_score(y_test, y_val_pred, zero_division=0), 
                             f1_score(y_val, y_val_pred, zero_division=0),
                             roc_auc_score(y_val, y_val_proba)
                             ]

print(results)

                Model  Accuracy  Precision    Recall        F1   ROC-AUC
0               Dummy  0.513333   0.410853  0.430894  0.420635  0.500758
1  LogisticRegression  0.530000   0.414286  0.686992  0.381579  0.506201


# Краткий текстовый отчет

## Чем `бейзлайн` отличается от `логистической регрессии` по качеству? 

Логическая регрессия заметно улучшает метрики, так как baseline, дает нижнюю планку качества 

## Насколько сильно выросла (или не выросла) `accuracy` и `ROC-AUC`?

В логической регресси `accuracy` выросло почти на 2%, а `ROC-AUC` чуть более, чем на 0.5%

## Как изменение регуляризации влияло на качество?

При уменщении регуляризации(увелечении C), модель становилась свободнее и могла переобучаться, при увеличении регуляризации она становилась проще и не могла переобучаться, по итогу самым удачным вариантом стало уменщение C, где модель сжали в ее переобучении

## Простые выводы о том, какая модель кажется разумной для этой задачи и почему.

Я сделал вывод о том, что для этой задачи разумнее будет использовать модель логистической регресии, потому что Dummy не берет информацию из признаков и ведет себя очень просто, также с Dummy, было бы не очень удобно учитывать таргеты из датасета которые влияют на результат 